# Illustrations for the movie GAIA, Enceladus and ... Sausage!

## Tidal interactions between galaxy and satellite/body

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation, FFMpegWriter

from IPython.display import Image, display

from pathlib import Path



# =========================

# Config

# =========================



OUT_DIR = Path("Infographics/Telemetry/media-site/animations/tidal_interaction")

OUT_DIR.mkdir(exist_ok=True)



WEBM_PATH = OUT_DIR / "tidal_interaction_galaxy_demo.webm"



FPS = 24

DURATION_SEC = 14

FRAMES = FPS * DURATION_SEC



FIGSIZE = (16, 9)

DPI = 120



BG = "#020510"

TEXT = "#c9d3df"

MUTED = "#6f8194"

CYAN = "#35c9ff"

ORANGE = "#ff9a3c"

RED = "#ff4d4d"

WHITE = "#f0f6ff"



rng = np.random.default_rng(501)



# =========================

# Synthetic galaxy particle field

# =========================



N_DISK = 9500

N_HALO = 2200

N_TAIL = 3200



# Main disk particles

r = rng.gamma(shape=2.0, scale=7.0, size=N_DISK)

r = np.clip(r, 0, 34)

theta = rng.uniform(0, 2 * np.pi, N_DISK)



# Initial disk

x0 = r * np.cos(theta)

y0 = 0.72 * r * np.sin(theta)



# Halo / outer diffuse particles

rh = rng.uniform(18, 45, N_HALO)

th = rng.uniform(0, 2 * np.pi, N_HALO)

xh0 = rh * np.cos(th)

yh0 = 0.80 * rh * np.sin(th)



# Tidal tail seed

u = rng.uniform(0, 1, N_TAIL)

tail_r = 10 + 38 * u

tail_width = 1.2 + 5.0 * u

tail_angle = rng.normal(0, 0.20, N_TAIL)



xt0 = tail_r * np.cos(tail_angle)

yt0 = tail_r * np.sin(tail_angle) + rng.normal(0, tail_width, N_TAIL)



# rotate tail base

tail_rot = -1.05

xt = xt0 * np.cos(tail_rot) - yt0 * np.sin(tail_rot)

yt = xt0 * np.sin(tail_rot) + yt0 * np.cos(tail_rot)



# Combined arrays

base_x = np.concatenate([x0, xh0, xt])

base_y = np.concatenate([y0, yh0, yt])

kind = np.concatenate([

    np.zeros(N_DISK),

    np.ones(N_HALO),

    np.full(N_TAIL, 2),

])



# Particle colors are mapped by local density/proxy

base_c = np.concatenate([

    np.clip(1.0 - r / 34, 0.15, 1.0),

    np.full(N_HALO, 0.12),

    np.clip(0.45 - u * 0.30, 0.08, 0.45),

])



# =========================

# Figure setup

# =========================



fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor(BG)

ax.set_facecolor(BG)



fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

ax.set_position([0, 0, 1, 1])



ax.set_xlim(-48, 48)

ax.set_ylim(-27, 27)

ax.set_aspect("equal")

ax.axis("off")



# =========================

# Artists

# =========================



scatter = ax.scatter(

    base_x,

    base_y,

    c=base_c,

    s=0.55,

    cmap="turbo",

    alpha=0.72,

    linewidths=0,

    vmin=0,

    vmax=1,

    zorder=5,

)



# central density / bar contours

contour_lines = []

for rad, alpha in [(3.2, 0.65), (5.2, 0.45), (7.6, 0.30)]:

    line, = ax.plot([], [], color=RED, linewidth=1.0, alpha=alpha, zorder=20)

    contour_lines.append((line, rad))



# satellite

satellite = ax.scatter([], [], s=140, color=ORANGE, edgecolor=WHITE, linewidths=0.8, alpha=0.0, zorder=30)

satellite_glow = ax.scatter([], [], s=700, color=ORANGE, alpha=0.0, linewidths=0, zorder=29)



# orbit / trajectory

traj_line, = ax.plot([], [], color=ORANGE, linewidth=1.0, linestyle=(0, (4, 5)), alpha=0.0, zorder=18)



# scale / title

title = ax.text(

    -45.5, 24.5,

    "TIDAL INTERACTION — DEMONSTRATION",

    color=TEXT,

    fontsize=18,

    fontweight="bold",

    ha="left",

    va="center",

    zorder=40,

)



subtitle = ax.text(

    -45.5, 22.2,

    "satellite passage distorts the disk and pulls out tidal material",

    color=MUTED,

    fontsize=11,

    fontweight="bold",

    ha="left",

    va="center",

    zorder=40,

)



time_text = ax.text(

    42.5, 24.2,

    "t = 1.00 Gyr",

    color=RED,

    fontsize=15,

    fontweight="bold",

    ha="right",

    va="center",

    zorder=40,

)



stage_text = ax.text(

    -45.5, -24.2,

    "stage: pre-encounter disk",

    color=CYAN,

    fontsize=12,

    ha="left",

    va="center",

    bbox=dict(

        boxstyle="round,pad=0.35",

        facecolor="#07111f",

        edgecolor="#2b4358",

        alpha=0.88,

    ),

    zorder=40,

)



# =========================

# Dynamics helpers

# =========================



def smoothstep(edge0, edge1, value):

    t = np.clip((value - edge0) / (edge1 - edge0), 0, 1)

    return t * t * (3 - 2 * t)



def satellite_position(t):

    """

    Schematic fly-by orbit: not a real integration.

    """

    phi = -2.4 + 4.5 * t

    rad = 34 - 18 * np.exp(-((t - 0.45) / 0.22) ** 2)

    sx = rad * np.cos(phi) + 6 * (t - 0.5)

    sy = 0.72 * rad * np.sin(phi) - 1.5

    return sx, sy



def transform_particles(t):

    """

    Demonstration-only deformation:

    - disk slowly forms a bar

    - one-armed/two-armed spiral distortion appears

    - tidal tail is pulled out after closest approach

    """

    x = base_x.copy()

    y = base_y.copy()



    rr = np.sqrt(x**2 + (y / 0.72)**2)

    ang = np.arctan2(y / 0.72, x)



    encounter = smoothstep(0.22, 0.45, t)

    post = smoothstep(0.42, 0.78, t)

    relax = smoothstep(0.80, 1.00, t)



    # Bar formation in inner disk

    bar_strength = 0.32 * encounter * (1.0 - 0.25 * relax)

    inner = kind == 0

    x[inner] *= 1 + bar_strength * np.exp(-(rr[inner] / 9) ** 2)

    y[inner] *= 1 - 0.45 * bar_strength * np.exp(-(rr[inner] / 9) ** 2)



    # Spiral winding / tidal phase shift

    twist = encounter * (0.55 + 1.15 * post)

    new_ang = ang + twist * np.exp(-rr / 26) * (rr / 13)



    # m=2 perturbation

    amp = 0.18 * encounter * np.exp(-rr / 32)

    rr2 = rr * (1 + amp * np.sin(2 * new_ang - 5.5 * t))



    # m=1 lopsided disturbance

    lopsided = 0.16 * post * np.exp(-rr / 36)

    x_shift = lopsided * rr * np.cos(new_ang - 2.5)

    y_shift = 0.72 * lopsided * rr * np.sin(new_ang - 2.5)



    disk_or_halo = kind < 2

    x[disk_or_halo] = rr2[disk_or_halo] * np.cos(new_ang[disk_or_halo]) + x_shift[disk_or_halo]

    y[disk_or_halo] = 0.72 * rr2[disk_or_halo] * np.sin(new_ang[disk_or_halo]) + y_shift[disk_or_halo]



    # Tidal tail reveal and stretching

    tail = kind == 2

    tail_a = smoothstep(0.33, 0.58, t)



    x[tail] = (

        base_x[tail] * (0.25 + 0.95 * tail_a)

        + 9.0 * tail_a * np.sin(2.2 * t + base_y[tail] * 0.03)

    )

    y[tail] = (

        base_y[tail] * (0.20 + 1.05 * tail_a)

        - 5.0 * tail_a

        + 4.0 * tail_a * np.sin(base_x[tail] * 0.045 + 3.0 * t)

    )



    # Overall slow rotation

    rot = 0.12 * np.sin(2 * np.pi * (t * 0.45))

    xr = x * np.cos(rot) - y * np.sin(rot)

    yr = x * np.sin(rot) + y * np.cos(rot)



    # Fade tail before it appears by pushing it behind alpha via colors/sizes later

    return xr, yr, tail_a, encounter, post



# =========================

# Animation update

# =========================



traj_x = []

traj_y = []



def update(frame):

    t = frame / (FRAMES - 1)



    x, y, tail_a, encounter, post = transform_particles(t)



    # Size / alpha modulation

    colors = base_c.copy()

    colors[kind == 2] *= tail_a



    scatter.set_offsets(np.column_stack([x, y]))

    scatter.set_array(colors)



    # alpha: tail is effectively invisible at start through color + global alpha

    scatter.set_alpha(0.70)



    # Satellite

    sx, sy = satellite_position(t)

    sat_a = smoothstep(0.08, 0.18, t) * (1.0 - smoothstep(0.88, 0.98, t))



    satellite.set_offsets([[sx, sy]])

    satellite.set_alpha(0.92 * sat_a)



    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 6 * t)

    satellite_glow.set_offsets([[sx, sy]])

    satellite_glow.set_alpha((0.10 + 0.10 * pulse) * sat_a)

    satellite_glow.set_sizes([520 + 240 * pulse])



    # Trajectory

    traj_x.append(sx)

    traj_y.append(sy)

    if len(traj_x) > 100:

        traj_x.pop(0)

        traj_y.pop(0)



    traj_line.set_data(traj_x, traj_y)

    traj_line.set_alpha(0.45 * sat_a)



    # Central bar / density contours

    contour_a = smoothstep(0.28, 0.52, t)

    angle = -0.25 + 0.75 * t

    u = np.linspace(0, 2 * np.pi, 220)



    for line, rad in contour_lines:

        bx = rad * (1.0 + 0.50 * encounter) * np.cos(u)

        by = rad * (0.62 - 0.22 * encounter) * np.sin(u)



        xr = bx * np.cos(angle) - by * np.sin(angle)

        yr = bx * np.sin(angle) + by * np.cos(angle)



        line.set_data(xr, yr)

        line.set_alpha(0.45 * contour_a)



    # Time label

    sim_time = 1.00 + 0.50 * t

    time_text.set_text(f"t = {sim_time:.2f} Gyr")



    if t < 0.24:

        stage_text.set_text("stage: pre-encounter rotating disk")

        stage_text.set_color(CYAN)

    elif t < 0.48:

        stage_text.set_text("stage: satellite fly-by begins tidal distortion")

        stage_text.set_color(ORANGE)

    elif t < 0.72:

        stage_text.set_text("stage: tidal tail and asymmetric spiral response")

        stage_text.set_color(RED)

    else:

        stage_text.set_text("stage: disturbed remnant with bar-like core")

        stage_text.set_color(WHITE)



    return (

        scatter,

        satellite,

        satellite_glow,

        traj_line,

        time_text,

        stage_text,

        *[line for line, _ in contour_lines],

    )



# =========================

# Save GIF

# =========================



anim = FuncAnimation(

    fig,

    update,

    frames=FRAMES,

    interval=1000 / FPS,

    blit=False

)



writer = FFMpegWriter(fps=FPS, codec="libvpx-vp9", extra_args=["-crf", "32", "-b:v", "0", "-pix_fmt", "yuv420p"])

anim.save(WEBM_PATH, writer=writer)



plt.close(fig)



print(f"Saved: {WEBM_PATH}")

print(f"Saved: {WEBM_PATH}")


## Mikly Way absorbs Sag dwarf galaxy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "milky_way_sagittarius_stream_demo_v2.gif"

FPS = 24
DURATION_SEC = 20
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
BLUE = "#0b63ff"
ORANGE = "#ff9a3c"
RED = "#ff4d4d"
WHITE = "#f0f6ff"
GREEN = "#48ffb3"

rng = np.random.default_rng(731)

# =========================
# Particle counts
# =========================

N_MW_DISK = 12000
N_MW_HALO = 2800
N_SGR_CORE = 1600
N_SGR_STREAM = 5200

# =========================
# Milky Way model
# =========================

r = rng.gamma(shape=2.0, scale=6.2, size=N_MW_DISK)
r = np.clip(r, 0, 34)
theta = rng.uniform(0, 2 * np.pi, N_MW_DISK)

spiral_phase = theta + 0.55 * r
arm_boost = 0.18 * np.sin(2 * spiral_phase)
r_mod = r * (1 + arm_boost)

mw_x0 = r_mod * np.cos(theta)
mw_y0 = 0.42 * r_mod * np.sin(theta)
mw_c = np.clip(1.0 - r / 34, 0.18, 1.0)

rh = rng.uniform(10, 46, N_MW_HALO)
th = rng.uniform(0, 2 * np.pi, N_MW_HALO)
halo_x0 = rh * np.cos(th)
halo_y0 = 0.78 * rh * np.sin(th)
halo_c = np.full(N_MW_HALO, 0.14)

# =========================
# Sagittarius dwarf model
# =========================

# Diffuse core particles
sgr_r = rng.rayleigh(scale=1.2, size=N_SGR_CORE)
sgr_th = rng.uniform(0, 2 * np.pi, N_SGR_CORE)
sgr_core_x0 = sgr_r * np.cos(sgr_th)
sgr_core_y0 = 0.70 * sgr_r * np.sin(sgr_th)

# Stream/debris particles
u = rng.uniform(0, 1, N_SGR_STREAM)

# Important: fixed random fields, so the stream does not boil randomly every frame
stream_along_noise = rng.normal(0, 1.0, N_SGR_STREAM)
stream_cross_noise = rng.normal(0, 1.0, N_SGR_STREAM)
stream_family = rng.integers(0, 5, N_SGR_STREAM)
stream_pull_noise = rng.random(N_SGR_STREAM)
stream_swirl_phase = rng.uniform(0, 2 * np.pi, N_SGR_STREAM)

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

ax.set_xlim(-48, 48)
ax.set_ylim(-27, 27)
ax.set_aspect("equal")
ax.axis("off")

# =========================
# Artists
# =========================

mw_scatter = ax.scatter(
    np.concatenate([mw_x0, halo_x0]),
    np.concatenate([mw_y0, halo_y0]),
    c=np.concatenate([mw_c, halo_c]),
    s=np.concatenate([
        np.full(N_MW_DISK, 0.50),
        np.full(N_MW_HALO, 0.28),
    ]),
    cmap="turbo",
    alpha=0.68,
    linewidths=0,
    vmin=0,
    vmax=1,
    zorder=5,
)

sgr_core_scatter = ax.scatter(
    [],
    [],
    s=1.2,
    color=CYAN,
    alpha=0.0,
    linewidths=0,
    zorder=20,
)

sgr_stream_scatter = ax.scatter(
    [],
    [],
    s=0.75,
    color=CYAN,
    alpha=0.0,
    linewidths=0,
    zorder=18,
)

sgr_glow = ax.scatter(
    [],
    [],
    s=1600,
    color=CYAN,
    alpha=0.0,
    linewidths=0,
    zorder=17,
)

nucleus_glow = ax.scatter(
    [0],
    [0],
    s=1400,
    color=ORANGE,
    alpha=0.13,
    linewidths=0,
    zorder=10,
)

nucleus = ax.scatter(
    [0],
    [0],
    s=120,
    color=WHITE,
    alpha=0.85,
    linewidths=0,
    zorder=11,
)

orbit_line, = ax.plot(
    [],
    [],
    color=CYAN,
    linewidth=1.0,
    linestyle=(0, (4, 5)),
    alpha=0.0,
    zorder=15,
)

title = ax.text(
    -45.5,
    24.5,
    "MILKY WAY — ENCELADUS DWARF INTERACTION",
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40,
)

subtitle = ax.text(
    -45.5,
    22.2,
    "a dwarf galaxy is tidally disrupted and stretched into a stellar stream",
    color=MUTED,
    fontsize=11,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40,
)

time_text = ax.text(
    43.5,
    24.2,
    "t = 0.00 Gyr",
    color=RED,
    fontsize=15,
    fontweight="bold",
    ha="right",
    va="center",
    zorder=40,
)

stage_text = ax.text(
    -45.5,
    -24.2,
    "stage: Sagittarius approaches the Milky Way halo",
    color=CYAN,
    fontsize=12,
    ha="left",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.88,
    ),
    zorder=40,
)

label_mw = ax.text(
    5.2,
    -4.4,
    "Milky Way",
    color=TEXT,
    fontsize=11,
    fontweight="bold",
    alpha=0.76,
    zorder=30,
)

label_sgr = ax.text(
    0,
    0,
    "Enceladus dwarf",
    color=CYAN,
    fontsize=10,
    fontweight="bold",
    alpha=0.0,
    zorder=30,
)

# =========================
# Helpers
# =========================

def smoothstep(edge0, edge1, value):
    t = np.clip((value - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)


def orbit_position(t):
    """
    Demonstration orbit:
    Sagittarius falls from the halo, wraps around the Milky Way,
    and is gradually drawn inward.
    """
    phi = -2.45 + 4.45 * t
    radius = 39 - 22 * smoothstep(0.20, 0.86, t)

    x = radius * np.cos(phi)
    y = 0.62 * radius * np.sin(phi)

    inward = smoothstep(0.64, 1.00, t)
    x *= (1.0 - 0.38 * inward)
    y *= (1.0 - 0.38 * inward)

    return x, y, phi, radius


def tangent_vector(phi):
    tx = -np.sin(phi)
    ty = 0.62 * np.cos(phi)
    norm = np.sqrt(tx * tx + ty * ty)
    return tx / norm, ty / norm


def make_mw(t):
    """
    Mild response of the Milky Way disk/halo.
    """
    all_x = np.concatenate([mw_x0, halo_x0])
    all_y = np.concatenate([mw_y0, halo_y0])

    rr = np.sqrt(all_x**2 + (all_y / 0.55) ** 2)
    ang = np.arctan2(all_y / 0.55, all_x)

    response = smoothstep(0.30, 0.88, t)
    warp = 0.08 * response * np.sin(ang - 3.8 * t) * np.exp(-rr / 35)

    x = all_x * (1.0 + 0.03 * response * np.sin(2 * ang))
    y = all_y + rr * warp

    return x, y


def make_sagittarius(t):
    sx, sy, phi, radius = orbit_position(t)

    disrupt = smoothstep(0.16, 0.58, t)
    stream_a = smoothstep(0.24, 0.62, t)
    inward = smoothstep(0.58, 1.00, t)

    # Почти полное исчезновение к концу
    vanish = 1.0 - smoothstep(0.88, 1.00, t)

    tx, ty = tangent_vector(phi)
    nx, ny = -ty, tx

    # Ядро Стрельца сначала вытягивается, потом растворяется
    core_stretch = 1.0 + 4.2 * disrupt
    core_compress = 1.0 - 0.50 * disrupt

    cx = sx + tx * sgr_core_x0 * core_stretch + nx * sgr_core_y0 * core_compress
    cy = sy + ty * sgr_core_x0 * core_stretch + ny * sgr_core_y0 * core_compress

    core_alpha = smoothstep(0.02, 0.10, t) * (1.0 - 0.92 * inward) * vanish

    # ==========================================================
    # Главное изменение:
    # debris строится как дугообразный рой вокруг Млечного Пути,
    # а не как прямая струя вдоль касательной.
    # ==========================================================

    q = np.abs(u - 0.5) * 2.0
    sign = np.where(u < 0.5, -1.0, 1.0)

    # Большая угловая протяжённость — поток идёт по дуге
    arc = sign * (0.20 + 2.40 * q) * stream_a

    # Несколько под-потоков / клочков, чтобы не было одной линии
    family_offset = (stream_family - 2.0) * 0.28 * stream_a

    # Индивидуальный разброс фазы по орбите
    phase_scatter = 0.28 * stream_along_noise * stream_a

    local_phi = phi + arc + family_offset + phase_scatter

    # Радиальный разброс: поток становится облаком, а не ниткой
    radial_spread = (
        3.5
        + 7.0 * stream_a
        + 4.0 * q * stream_a
    ) * stream_cross_noise

    # Материал постепенно втягивается внутрь
    pull = inward * (0.22 + 0.70 * q + 0.20 * stream_pull_noise)

    local_radius = radius * (1.0 - 0.56 * pull) + radial_spread

    # Основная дуга вокруг центра Млечного Пути
    stream_x = local_radius * np.cos(local_phi)
    stream_y = 0.62 * local_radius * np.sin(local_phi)

    # Дополнительная фазовая мешанина: облако рваное, не гладкая кривая
    swirl = 1.8 * stream_a * np.sin(
        4.0 * q + 2.5 * t + stream_swirl_phase
    )

    stx, sty = tangent_vector(local_phi)
    snx, sny = -sty, stx

    stream_x += snx * swirl + stx * 0.9 * stream_a * stream_cross_noise
    stream_y += sny * swirl + sty * 0.9 * stream_a * stream_cross_noise

    # В конце возвращаем более красивое "всасывание":
    # поток не просто исчезает, а сжимается к внутреннему гало/диску
    sink = smoothstep(0.68, 1.00, t)

    stream_x *= (1.0 - 0.42 * sink)
    stream_y *= (1.0 - 0.52 * sink)

    stream_alpha = stream_a * vanish

    return sx, sy, cx, cy, core_alpha, stream_x, stream_y, stream_alpha
    
def update(frame):
    t = frame / (FRAMES - 1)

    # Milky Way response
    mx, my = make_mw(t)
    mw_scatter.set_offsets(np.column_stack([mx, my]))

    # Enceladus dwarf + stream
    sx, sy, cx, cy, core_alpha, stream_x, stream_y, stream_alpha = make_sagittarius(t)

    sgr_core_scatter.set_offsets(np.column_stack([cx, cy]))
    sgr_core_scatter.set_alpha(0.62 * core_alpha)

    sgr_stream_scatter.set_offsets(np.column_stack([stream_x, stream_y]))
    sgr_stream_scatter.set_alpha(0.52 * stream_alpha)

    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 6 * t)

    sgr_glow.set_offsets([[sx, sy]])
    sgr_glow.set_alpha((0.07 + 0.06 * pulse) * core_alpha)
    sgr_glow.set_sizes([1250 + 520 * pulse])

    # Orbit trace
    orbit_trace_x.append(sx)
    orbit_trace_y.append(sy)

    if len(orbit_trace_x) > 220:
        orbit_trace_x.pop(0)
        orbit_trace_y.pop(0)

    orbit_line.set_data(orbit_trace_x, orbit_trace_y)
    orbit_line.set_alpha(
        0.35
        * smoothstep(0.04, 0.18, t)
        * (1.0 - smoothstep(0.78, 0.98, t))
    )

    # Sagittarius label follows core while it exists
    label_sgr.set_position((sx + 2.0, sy + 1.6))
    label_sgr.set_alpha(0.88 * core_alpha)

    sim_time = 0.00 + 3.5 * t
    time_text.set_text(f"t = {sim_time:.2f} Gyr")

    if t < 0.25:
        stage_text.set_text("stage: Sagittarius approaches the Milky Way halo")
        stage_text.set_color(CYAN)
    elif t < 0.52:
        stage_text.set_text("stage: tidal forces stretch the dwarf galaxy")
        stage_text.set_color(ORANGE)
    elif t < 0.78:
        stage_text.set_text("stage: stars are stripped into a broad stream")
        stage_text.set_color(RED)
    else:
        stage_text.set_text("stage: disrupted debris is absorbed into the Milky Way")
        stage_text.set_color(GREEN)

    return (
        mw_scatter,
        sgr_core_scatter,
        sgr_stream_scatter,
        sgr_glow,
        orbit_line,
        label_sgr,
        time_text,
        stage_text,
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "bar_strength_dark_theme.gif"

FPS = 24
DURATION_SEC = 10
FRAMES = FPS * DURATION_SEC

FIGSIZE = (9, 5.6)
DPI = 140

# Dark theme colors (our style)
BG = "#020510"
GRID = "#2b4358"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
RED = "#ff5a45"
WHITE = "#f0f6ff"

# =========================
# Synthetic data (same as before)
# =========================

t_anchor = np.array([
    0.00, 0.15, 0.30, 0.45, 0.60, 0.75, 0.88, 1.00,
    1.12, 1.25, 1.40, 1.60, 1.78, 1.95, 2.12, 2.30,
    2.50, 2.75, 3.00, 3.18, 3.35, 3.55, 3.80, 4.05,
    4.30, 4.55, 4.80, 5.05, 5.35, 5.70, 6.05, 6.40,
    6.75, 7.10, 7.45, 7.80, 8.15, 8.50, 8.85, 9.20,
    9.55, 10.00
])

a2_anchor = np.array([
    0.00, 0.015, 0.045, 0.095, 0.22, 0.39, 0.17, 0.32,
    0.16, 0.18, 0.21, 0.18, 0.22, 0.16, 0.23, 0.19,
    0.23, 0.22, 0.25, 0.27, 0.24, 0.18, 0.19, 0.21,
    0.20, 0.21, 0.22, 0.225, 0.235, 0.24, 0.245, 0.25,
    0.255, 0.265, 0.272, 0.278, 0.285, 0.292, 0.305, 0.318,
    0.328, 0.34
])

omega_anchor = np.array([
    np.nan, 24.5, 26.9, 25.6, 26.2, 25.0, 25.4, 24.4,
    24.8, 24.6, 24.4, 24.2, 24.0, 23.6, 23.2, 22.8,
    22.2, 21.7, 21.0, 20.6, 20.4, 19.5, 19.3, 19.0,
    18.7, 18.3, 17.9, 17.3, 16.9, 16.5, 16.0, 15.6,
    15.1, 14.7, 14.2, 13.8, 13.3, 12.9, 12.4, 11.8,
    11.3, 10.2
])

t = np.linspace(0, 10, 900)
a2 = np.interp(t, t_anchor, a2_anchor)

valid = ~np.isnan(omega_anchor)
omega = np.interp(t, t_anchor[valid], omega_anchor[valid])

rng = np.random.default_rng(12)

a2 += np.interp(t, np.linspace(0, 10, 90), rng.normal(0, 0.007, 90))
a2 = np.clip(a2, 0, 0.6)

omega += np.interp(t, np.linspace(0, 10, 55), rng.normal(0, 0.18, 55))
omega = np.clip(omega, 9, 28)

# =========================
# Figure
# =========================

fig, ax1 = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax1.set_facecolor(BG)

ax2 = ax1.twinx()

ax1.set_xlim(0, 10)
ax1.set_ylim(0, 0.6)
ax2.set_ylim(9, 28)

# Axes styling
for ax in (ax1, ax2):
    ax.tick_params(colors=TEXT, labelsize=11, direction="in")

for spine in ax1.spines.values():
    spine.set_color(GRID)

for spine in ax2.spines.values():
    spine.set_color(GRID)

ax1.set_xlabel("t [Gyr]", color=TEXT, fontsize=13)
ax1.set_ylabel("A₂", color=CYAN, fontsize=13)
ax2.set_ylabel("Ωₚ [km/s/kpc]", color=RED, fontsize=13)

# Subtle grid
ax1.grid(color=GRID, alpha=0.25, linewidth=0.6)

# =========================
# Lines
# =========================

line_a2, = ax1.plot([], [], color=CYAN, linewidth=1.6)
line_omega, = ax2.plot([], [], color=RED, linewidth=1.4)

# Glow (cinematic effect)
glow_a2, = ax1.plot([], [], color=CYAN, linewidth=6, alpha=0.08)
glow_omega, = ax2.plot([], [], color=RED, linewidth=5, alpha=0.06)

# Cursor
cursor_a2, = ax1.plot([], [], "o", color=CYAN, ms=4)
cursor_omega, = ax2.plot([], [], "o", color=RED, ms=4)

# HUD text
time_text = ax1.text(
    0.02, 0.95, "",
    transform=ax1.transAxes,
    color=TEXT,
    fontsize=12,
    ha="left",
    va="top"
)

# =========================
# Animation
# =========================

def ease(x):
    return 1 - (1 - x)**3

def update(frame):
    p = ease(frame / (FRAMES - 1))
    xmax = p * 10

    mask = t <= xmax

    tv = t[mask]
    a2v = a2[mask]
    ov = omega[mask]

    line_a2.set_data(tv, a2v)
    line_omega.set_data(tv, ov)

    glow_a2.set_data(tv, a2v)
    glow_omega.set_data(tv, ov)

    if len(tv) > 0:
        cursor_a2.set_data([tv[-1]], [a2v[-1]])
        cursor_omega.set_data([tv[-1]], [ov[-1]])
        time_text.set_text(f"t = {tv[-1]:.2f} Gyr")

    return (
        line_a2, line_omega,
        glow_a2, glow_omega,
        cursor_a2, cursor_omega,
        time_text
    )

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "bar_strength_scatter_dark.gif"

FPS = 24
DURATION_SEC = 10
FRAMES = FPS * DURATION_SEC

FIGSIZE = (10, 6)
DPI = 140

BG = "#020510"
GRID = "#2b4358"
TEXT = "#c9d3df"
MUTED = "#6f8194"

RED = "#ff5a45"

rng = np.random.default_rng(42)

# =========================
# Synthetic structure
# =========================

labels = ["1/20","1/10","1/8","1/6","1/5","1/4","1/3","1/2","1/1"]
x_pos = np.arange(len(labels))

POINTS_PER_COL = 45

all_x = []
all_y = []
all_color = []
all_marker = []

for i, x in enumerate(x_pos):
    for _ in range(POINTS_PER_COL):
        # decreasing trend
        base = 0.35 - 0.03 * i
        y = np.clip(base + rng.normal(0, 0.06), 0, 0.5)

        # color ~ impact parameter b
        b = rng.uniform(-100, 100)

        # crosses for some high m/M
        is_cross = (i >= 6) and (rng.random() < 0.4)

        all_x.append(x + rng.normal(0, 0.22))
        all_y.append(y)
        all_color.append(b)
        all_marker.append("x" if is_cross else "o")

all_x = np.array(all_x)
all_y = np.array(all_y)
all_color = np.array(all_color)
all_marker = np.array(all_marker)

# =========================
# Shuffle points (important)
# =========================

idx = rng.permutation(len(all_x))

all_x = all_x[idx]
all_y = all_y[idx]
all_color = all_color[idx]
all_marker = all_marker[idx]

# =========================
# Figure
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.set_xlim(-0.6, len(labels)-0.4)
ax.set_ylim(0, 0.5)

# grid
for x in x_pos:
    ax.axvline(x, color=GRID, alpha=0.35, linewidth=1)

ax.grid(axis="y", color=GRID, alpha=0.25)

# weak-bar region
ax.axhspan(0, 0.15, color="#ffcc66", alpha=0.12)

# red reference line
ax.axhline(0.34, color=RED, linewidth=1.5)

# ticks
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, color=TEXT)

ax.set_ylabel("A₂", color=TEXT)
ax.set_xlabel("m / M", color=TEXT)

ax.tick_params(colors=TEXT)

for spine in ax.spines.values():
    spine.set_color(GRID)

# colorbar
sc = ax.scatter([], [], c=[], cmap="turbo", vmin=-100, vmax=100)
cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("b [kpc]", color=TEXT)
cbar.ax.yaxis.set_tick_params(color=TEXT)
plt.setp(cbar.ax.get_yticklabels(), color=TEXT)

# =========================
# Artists
# =========================

points_o = ax.scatter([], [], s=20, cmap="turbo", vmin=-100, vmax=100)
points_x = ax.scatter([], [], s=22, cmap="turbo", vmin=-100, vmax=100, marker="x")

time_text = ax.text(
    0.02, 0.95, "",
    transform=ax.transAxes,
    color=TEXT,
    fontsize=12,
    ha="left",
    va="top"
)

# =========================
# Animation
# =========================

def update(frame):
    p = frame / (FRAMES - 1)
    n = int(p * len(all_x))

    x = all_x[:n]
    y = all_y[:n]
    c = all_color[:n]
    m = all_marker[:n]

    mask_o = m == "o"
    mask_x = m == "x"

    points_o.set_offsets(np.column_stack([x[mask_o], y[mask_o]]))
    points_o.set_array(c[mask_o])

    points_x.set_offsets(np.column_stack([x[mask_x], y[mask_x]]))
    points_x.set_array(c[mask_x])

    time_text.set_text(f"models sampled: {n}")

    return points_o, points_x, time_text

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000/FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "milky_way_enceladus_absorption.gif"

FPS = 24
DURATION_SEC = 20
FRAMES = FPS * DURATION_SEC

MODE = "curved_cloud"  # "compact" or "curved_cloud"

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
ORANGE = "#ff9a3c"
WHITE = "#f0f6ff"
RED = "#ff5a45"

rng = np.random.default_rng(731)

# =========================
# Particle counts
# =========================

N_MW = 14000
N_ENC_CORE = 1200
N_ENC_STREAM = 2600

# =========================
# Milky Way — disk + halo + spiral structure
# =========================

N_MW_DISK = 12000
N_MW_HALO = 2800

# Disk
r = rng.gamma(shape=2.0, scale=6.2, size=N_MW_DISK)
r = np.clip(r, 0, 34)
theta = rng.uniform(0, 2 * np.pi, N_MW_DISK)

spiral_phase = theta + 0.55 * r
arm_boost = 0.18 * np.sin(2 * spiral_phase)
r_mod = r * (1 + arm_boost)

mw_disk_x0 = r_mod * np.cos(theta)
mw_disk_y0 = 0.42 * r_mod * np.sin(theta)
mw_disk_c = np.clip(1.0 - r / 34, 0.18, 1.0)

# Halo
rh = rng.uniform(10, 46, N_MW_HALO)
th = rng.uniform(0, 2 * np.pi, N_MW_HALO)

mw_halo_x0 = rh * np.cos(th)
mw_halo_y0 = 0.78 * rh * np.sin(th)
mw_halo_c = np.full(N_MW_HALO, 0.14)

mw_x0 = np.concatenate([mw_disk_x0, mw_halo_x0])
mw_y0 = np.concatenate([mw_disk_y0, mw_halo_y0])
mw_c = np.concatenate([mw_disk_c, mw_halo_c])
mw_s = np.concatenate([
    np.full(N_MW_DISK, 0.50),
    np.full(N_MW_HALO, 0.28),
])

# =========================
# Enceladus initial blob
# =========================

enc_r = rng.rayleigh(1.2, N_ENC_CORE)
enc_th = rng.uniform(0, 2*np.pi, N_ENC_CORE)

enc_x0 = enc_r * np.cos(enc_th)
enc_y0 = 0.7 * enc_r * np.sin(enc_th)

u = rng.uniform(0, 1, N_ENC_STREAM)
noise = rng.normal(0, 1, N_ENC_STREAM)

# =========================
# Figure
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

ax.set_xlim(-50, 50)
ax.set_ylim(-30, 30)
ax.set_aspect("equal")
ax.axis("off")

# Milky Way disk + halo
mw = ax.scatter(
    mw_x0,
    mw_y0,
    c=mw_c,
    cmap="turbo",
    s=mw_s,
    alpha=0.68,
    linewidths=0,
    vmin=0,
    vmax=1,
    zorder=5,
)

# Central nucleus
nucleus_glow = ax.scatter(
    [0],
    [0],
    s=1400,
    color=ORANGE,
    alpha=0.13,
    linewidths=0,
    zorder=10,
)

nucleus = ax.scatter(
    [0],
    [0],
    s=120,
    color=WHITE,
    alpha=0.85,
    linewidths=0,
    zorder=11,
)

enc_core = ax.scatter([], [], s=1.2, color=CYAN, alpha=0, zorder=20)
enc_stream = ax.scatter([], [], s=0.7, color=CYAN, alpha=0, zorder=18)


# =========================
# Labels
# =========================

ax.text(-45, 25, "MILKY WAY — ENCELADUS ACCRETION", color=TEXT, fontsize=18)

stage_text = ax.text(
    -45, -25,
    "",
    color=CYAN,
    fontsize=12
)

label_enc = ax.text(0,0,"Enceladus dwarf", color=CYAN, alpha=0)

# =========================
# Helpers
# =========================

def smoothstep(a,b,x):
    t = np.clip((x-a)/(b-a),0,1)
    return t*t*(3-2*t)

def orbit(t):
    phi = -2.4 + 4.4*t
    r = 40 - 25*smoothstep(0.2,0.9,t)
    return r*np.cos(phi), 0.6*r*np.sin(phi), phi, r

# =========================
# Enceladus model
# =========================

def enceladus(t):
    x, y, phi, r = orbit(t)

    disrupt = smoothstep(0.20, 0.58, t)
    inward = smoothstep(0.55, 1.00, t)
    vanish = 1.0 - smoothstep(0.88, 1.00, t)

    # --- базовые направления ---
    tx = -np.sin(phi)
    ty = 0.6 * np.cos(phi)
    norm = np.sqrt(tx * tx + ty * ty)
    tx, ty = tx / norm, ty / norm

    # к центру Галактики
    cx_dir = -x
    cy_dir = -y
    cn = np.sqrt(cx_dir**2 + cy_dir**2)
    cx_dir /= cn
    cy_dir /= cn

    # =========================
    # Core (карлик)
    # =========================

    stretch = 1.0 + 3.0 * disrupt
    compress = 1.0 - 0.35 * disrupt

    cx = x + cx_dir * enc_x0 * stretch + tx * enc_y0 * compress
    cy = y + cy_dir * enc_x0 * stretch + ty * enc_y0 * compress

    core_alpha = (1.0 - 0.85 * inward) * vanish

    # =========================
    # ONE stream only (ключ)
    # =========================

    q = u  # 0 → карлик, 1 → центр

    # убираем симметрию: оставляем только одну "сторону"
    # (раньше sign = ±1 создавал второй поток)
    # теперь ВСЕ частицы идут в одном направлении

    bend = 0.35 + 0.45 * disrupt

    # кривая траектория к центру
    path_x = (
        (1 - q) * x
        + q * (0.12 * x)
        + bend * np.sin(np.pi * q) * tx * 5.0
    )

    path_y = (
        (1 - q) * y
        + q * (0.12 * y)
        + bend * np.sin(np.pi * q) * ty * 3.2
    )

    # ширина потока
    width = (1.6 * (1 - q) + 0.35) * disrupt

    # перпендикуляр к направлению падения
    perp_x = -cy_dir
    perp_y = cx_dir

    # асимметричный шум (ВАЖНО: убираем вторую струю)
    one_sided_noise = np.abs(noise)  # только в одну сторону

    # "жидкостная" структура
    clump = 0.55 * np.sin(6.0 * q + 3.0 * t + noise)

    sx = path_x + perp_x * one_sided_noise * width + tx * clump * disrupt
    sy = path_y + perp_y * one_sided_noise * width + ty * clump * disrupt

    # втягивание
    sink = smoothstep(0.60, 1.00, t)

    sx *= (1.0 - 0.30 * sink * q)
    sy *= (1.0 - 0.40 * sink * q)

    stream_alpha = 0.65 * disrupt * vanish

    return x, y, cx, cy, core_alpha, sx, sy, stream_alpha
    


# =========================
# Animation
# =========================

def update(frame):
    t = frame/(FRAMES-1)

    x,y,cx,cy,a,sx,sy,sa = enceladus(t)

    enc_core.set_offsets(np.column_stack([cx,cy]))
    enc_core.set_alpha(a)

    enc_stream.set_offsets(np.column_stack([sx,sy]))
    enc_stream.set_alpha(sa)

    label_enc.set_position((x+2,y+2))
    label_enc.set_alpha(a)

    if t<0.3:
        stage_text.set_text("approach")
    elif t<0.6:
        stage_text.set_text("tidal disruption")
    else:
        stage_text.set_text("accretion into Milky Way")

    return enc_core, enc_stream, label_enc, stage_text

anim = FuncAnimation(fig, update, frames=FRAMES)

anim.save(GIF_PATH, writer=PillowWriter(fps=FPS))

plt.close(fig)
display(Image(filename=str(GIF_PATH)))
print("Saved:", GIF_PATH)